# 局部特征实践课：FAST · Harris · BRIEF · ORB · SIFT

**Local Features Lab — library call vs. step-by-step implementation, on real images with ground truth**

数据来自 **Oxford Affine Covariant Features** 数据集（Mikolajczyk et al., IJCV 2005）中的 9 张真实照片，每对图像附带官方**真值单应矩阵** `H1toN`：
把图 1 中的点 $(x, y, 1)^T$ 左乘 $H$ 再除以第三个分量，就得到它在图 N 中的位置。所以**每个关键点是否被重复检测到、每对匹配是对是错，都能用真值精确判断**。

| 序列 | 文件 | 变化 |
|---|---|---|
| graf（涂鸦墙） | graf_img1 / 3 / 4 / 5 | 视角逐渐增大 |
| bark（树皮） | bark_img1 / 4 / 6 | 缩放 + 旋转 |
| leuven（街景） | leuven_img1 / 6 | 光照变暗 |

此外可以用 `rotate_with_H` 把图像旋转任意角度，并同时得到旋转的真值矩阵。

**本课任务：**

* **FAST、BRIEF / ORB**：只需**调用现成函数**（`cv2.FastFeatureDetector`、`cv2.ORB`）完成检测与匹配；
* **Harris、SIFT**：做两种实现再对比——
  * **(A) 调用现成函数**：`cv2.cornerHarris`、`cv2.SIFT`；
  * **(B) 分步自己实现**：按课件步骤，只用 NumPy 和基础 OpenCV 运算（高斯滤波、Sobel、`dilate` 等），**不调用对应的现成检测器 / 描述子**；SIFT 步骤较多，你只需自己实现第 ① 步（尺度空间与 DoG 极值），其余步骤代码已给出；
  * **(C) 对比**：结果是否一致、性能（可重复性 / 正确匹配数）差多少、差在哪一步。

**要求**：下课前完成全部任务并提交本 Notebook（提交前 *Restart & Run All*，保留输出）。每个"问题"请**结合你自己运行得到的具体数字**回答。标 ★ 的小题为选做。

| 任务 | (A) 调用 | (B) 自己实现 | (C) 对比与实验 |
|---|---|---|---|
| 1 FAST | `FastFeatureDetector` | — | — |
| 2 Harris | `cornerHarris` | Sobel → 结构张量 → $R=\det M-\alpha\,\mathrm{tr}^2M$ | 响应相关性；旋转 / 缩放 / 光照下的可重复性 |
| 3 BRIEF 与 ORB | `ORB_create`；ORB 描述子方向固定为 0 即 BRIEF | — | — |
| 4 SIFT | `SIFT_create` | 自己写 ①：高斯金字塔 → DoG → 26 邻域极值 | 关键点数与速度；去边缘 / 去低对比的作用；7 组测试的正确匹配数；尺度比、方向差与真值对比 |
| 5 总结 | | | |

## 0. 公共工具（已给出，直接运行）

约定：坐标 $(x, y)$，$x$ 向右、$y$ 向下；数组索引 `img[y, x]`。
匹配统一使用"最近邻 + Lowe 比率检验（0.8）"：`match_ratio`；正确与否统一用真值判断（映射误差 < 5 像素）：`count_correct`。

In [ ]:
import os, sys, time
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
print('Python', sys.version.split()[0], '| OpenCV', cv2.__version__, '| NumPy', np.__version__)
assert hasattr(cv2, 'SIFT_create'), '需要 OpenCV >= 4.4（Python 3.7 可安装 opencv-python-headless 4.5/4.6）'

DATA = 'data'
for cand in ['data', os.path.join('..', 'data'), os.path.join('学生包', 'data')]:
    if os.path.isdir(cand):
        DATA = cand
        break
print('数据目录:', os.path.abspath(DATA))

def load_gray(name):
    img = cv2.imread(os.path.join(DATA, name + '.png'), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError('读不到 ' + name)
    return img

def load_H(seq, j):
    return np.loadtxt(os.path.join(DATA, '%s_H1to%d.txt' % (seq, j)))

def apply_H(H, pts):
    # 把 N x 2 的点用 3x3 单应矩阵映射（含齐次坐标除法）
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    q = np.hstack([pts, np.ones((len(pts), 1))]) @ H.T
    return q[:, :2] / q[:, 2:3]

def rotate_with_H(img, angle):
    # 绕图像中心旋转 angle 度（OpenCV 约定：正值 = 画面逆时针），画布扩展；返回 (旋转图, 3x3 真值矩阵)
    h, w = img.shape[:2]
    c = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(c, angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2.0 - c[0]
    M[1, 2] += nh / 2.0 - c[1]
    return cv2.warpAffine(img, M, (nw, nh)), np.vstack([M, [0, 0, 1]])

def inside(pts, shape, m):
    pts = np.asarray(pts, float).reshape(-1, 2); h, w = shape[:2]
    return (pts[:, 0] >= m) & (pts[:, 0] < w - m) & (pts[:, 1] >= m) & (pts[:, 1] < h - m)

def kp_xy(kps):
    return np.array([k.pt for k in kps], np.float64).reshape(-1, 2)

def repeatability(p1, p2, H, shape2, eps=3.0):
    # 图1的关键点映射到图2后（只统计落在图2内的），有多大比例在 eps 像素内能找到图2的关键点
    q = apply_H(H, p1)
    q = q[inside(q, shape2, 10)]
    if len(q) == 0 or len(p2) == 0:
        return 0.0
    d = np.sqrt(((q[:, None, :] - np.asarray(p2, float)[None, :, :]) ** 2).sum(-1)).min(1)
    return float((d < eps).mean())

def match_ratio(d1, d2, norm, ratio=0.8):
    # 最近邻 + Lowe 比率检验；二值描述子(uint8 打包) 用 cv2.NORM_HAMMING，浮点描述子用 cv2.NORM_L2
    out = []
    if d1 is None or d2 is None or len(d1) < 2 or len(d2) < 2:
        return out
    for m in cv2.BFMatcher(norm).knnMatch(d1, d2, k=2):
        if len(m) == 2 and m[0].distance < ratio * m[1].distance:
            out.append((m[0].queryIdx, m[0].trainIdx))
    return out

def count_correct(p1, p2, matches, H, eps=5.0):
    # p1, p2: 两图关键点坐标 (N x 2)；matches: [(i, j)]；返回 (正确数, 匹配总数, 每对是否正确)
    if len(matches) == 0:
        return 0, 0, np.zeros(0, bool)
    a = np.asarray(p1)[[i for i, j in matches]]; b = np.asarray(p2)[[j for i, j in matches]]
    ok = np.linalg.norm(apply_H(H, a) - b, axis=1) < eps
    return int(ok.sum()), len(matches), ok

def show(imgs, titles=None, size=3.2):
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size), squeeze=False)
    for i, im in enumerate(imgs):
        axes[0, i].imshow(im, cmap='gray'); axes[0, i].set_axis_off()
        if titles: axes[0, i].set_title(titles[i])
    plt.tight_layout(); plt.show()

names = ['graf_img1', 'graf_img3', 'graf_img4', 'graf_img5', 'bark_img1', 'bark_img4', 'bark_img6', 'leuven_img1', 'leuven_img6']
show([load_gray(n) for n in names[:4]], names[:4])
show([load_gray(n) for n in names[4:]], names[4:])
g1 = load_gray('graf_img1')

## 任务 1　FAST 角点检测（调用）

用 `cv2.FastFeatureDetector_create(threshold=20, nonmaxSuppression=...)` 在 `graf_img1` 上检测角点：
1. 分别统计**不做**与**做**非极大值抑制（NMS）时的角点数；
2. 把做 NMS 后的角点画在图上。

In [ ]:
# TODO：FAST 检测（不做 / 做 NMS 的角点数），并把 NMS 后的角点画出来

## 任务 2　Harris 角点（调用 + 自己实现 + 对比）

**(A) 调用**：`cv2.cornerHarris(img_float32, blockSize=5, ksize=3, k=0.05)`。

**(B) 自己实现** `harris_response(img, sigma=1.5, alpha=0.05)`：Sobel（`ksize=3`）求 $I_x, I_y$ → 高斯加权得
$A=\sum wI_x^2,\ B=\sum wI_xI_y,\ C=\sum wI_y^2$ → $R=AC-B^2-\alpha(A+C)^2$。

两种响应图都用同一个函数 `corners_from_R(R, K=500)` 取角点：7×7 非极大值抑制、$R>0.001R_{max}$、按 R 取前 K 个。

**(C) 对比**：
1. 两种 R 的相关系数；两组角点中有多少是同一位置（3 像素内）？
2. 两种实现分别计算可重复性：graf_img1 旋转 0°、15°、30°、45°、90°；bark 1→4（缩放约 2.5 倍）、bark 1→6（约 4 倍）、leuven 1→6（变暗）。

In [ ]:
def harris_response(img, sigma=1.5, alpha=0.05):
    # TODO (B)：不许调用 cv2.cornerHarris
    raise NotImplementedError

def corners_from_R(R, K=500):
    # TODO：7x7 NMS（提示：cv2.dilate），阈值 0.001*max，取前 K 个，返回 K x 2 的 (x, y)
    raise NotImplementedError

In [ ]:
# ---- (C) 对比 1（直接运行）----
harris_mine = lambda im: corners_from_R(harris_response(im))
harris_cv = lambda im: corners_from_R(cv2.cornerHarris(im.astype(np.float32), 5, 3, 0.05))
R = harris_response(g1); Rcv = cv2.cornerHarris(g1.astype(np.float32), 5, 3, 0.05)
print('R 的相关系数 = %.3f' % np.corrcoef(R.ravel(), Rcv.ravel())[0, 1])
pm, pc = harris_mine(g1), harris_cv(g1)
print('前 500 个角点中位置相同（3 像素内）的比例 = %.3f' % repeatability(pm, pc, np.eye(3), g1.shape))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
ax[0].imshow(np.sign(R) * np.abs(R) ** 0.25, cmap='bwr'); ax[0].set_title('own R (red>0 corner, blue<0 edge)'); ax[0].axis('off')
ax[1].imshow(g1, cmap='gray'); ax[1].plot(pm[:, 0], pm[:, 1], 'r+', ms=6, label='own'); ax[1].plot(pc[:, 0], pc[:, 1], 'yx', ms=4, label='cv2')
ax[1].legend(); ax[1].axis('off'); plt.show()

In [ ]:
# TODO (C) 对比 2：两种实现的可重复性表（旋转 0/15/30/45/90；bark 1->4、bark 1->6、leuven 1->6）

**问题 1**：(a) 两种实现的 R 为什么相关却不完全相同（提示：比较两者的窗口函数）？这对可重复性有影响吗？
(b) 旋转下可重复性大约多少？用 $M$ 的特征值解释为什么 Harris 旋转不变；为什么 90° 比 45° 更高（提示：90° 旋转需要插值吗）？
(c) bark 1→4 的可重复性降到多少？为什么固定窗口的 Harris 不具备尺度不变性？

> 你的回答：

## 任务 3　BRIEF 与 ORB（调用）

ORB = oFAST + rBRIEF。OpenCV 主库没有单独的 BRIEF：**把 ORB 关键点的方向固定为 0 再计算描述子，就是 BRIEF**（在 `detect` 与 `compute` 之间把每个关键点的 `k.angle` 设为 0）。

1. 用 `cv2.ORB_create(nfeatures=1000)` 在 `graf_img1` 和它旋转 30° 后的图像上分别检测关键点、计算描述子；
2. 用 `match_ratio(d1, d2, cv2.NORM_HAMMING)` 匹配，用 `count_correct` 统计正确匹配数 / 匹配总数，并画出匹配（绿 = 正确，红 = 错误）；
3. 把方向固定为 0（即 BRIEF），重复第 2 步。

In [ ]:
g30, H30 = rotate_with_H(g1, 30)
# TODO：ORB 与 BRIEF（方向固定为 0）在 graf_img1 / 旋转 30° 上的匹配

## 任务 4　SIFT（调用 + 自己实现 + 对比）

**(A) 调用**：`cv2.SIFT_create(nfeatures=1000)` 的 `detectAndCompute`。

**(B) 分步实现**（不调用 `cv2.SIFT_create`）。SIFT 步骤较多，本课**你只需要自己写第 ① 步**（尺度空间与 DoG 极值，这是 SIFT 尺度不变性的来源）；②③④ 的代码已给出（带注释），请读懂后运行，`my_sift` 把各步串起来：

| 步骤 | 函数 | 谁来写 | 要点 |
|---|---|---|---|
| ① 高斯尺度空间 | `gaussian_pyramid(img, n_oct=5, s=3, sigma0=1.6)` | **你** | 图像先除以 255；初始模糊 $\sqrt{1.6^2-0.5^2}$；每组 $s+3=6$ 层，$\sigma_i=\sigma_0k^i$，$k=2^{1/s}$，**逐层增量模糊** $\sqrt{\sigma_i^2-\sigma_{i-1}^2}$；下一组 = 本组第 $s$ 层隔点降采样 |
| ① DoG | `dog_pyramid(pyr)` | **你** | 相邻两层相减，每组 5 张 |
| ① 26 邻域极值 | `find_extrema(D, thr=0.5*0.04/3)` | **你** | 在中间 3 层上找比 3×3×3 邻域都大或都小、且 $\lvert D\rvert>$ thr 的点；返回 `(层, y, x)` 数组（提示：`cv2.dilate` / `cv2.erode` 求每层 3×3 邻域最大 / 最小，再在相邻三层间取最大 / 最小） |
| ② 精确定位 | `refine(D, ext, contrast, r)` | 已给出 | 二阶泰勒展开求亚像素偏移 $\hat{x}=-H^{-1}g$；去低对比度；用 2×2 Hessian 的 $\operatorname{tr}^2/\det$ 去边缘 |
| ③ 方向赋值 | `orientation(L, x, y, sigma)` | 已给出 | 36 bin 梯度方向直方图，取峰值为主方向 |
| ④ 描述子 | `descriptor(L, x, y, sigma, theta)` | 已给出 | 旋转到主方向后，4×4 子块 × 8 方向 = 128 维；归一化 → 截断 0.2 → 再归一化 |

★ 选做：自己重写 ② 或 ③ 或 ④，与给出的版本比较。

**(C) 对比**（两种实现都要做）：
1. 关键点数、耗时；把两种关键点画在 graf_img1 上（圆半径 ∝ 尺度）；
2. 下面 7 组测试的正确匹配数 / 匹配总数：graf_img1 旋转 30°、graf 1→3、1→4、1→5、bark 1→4、1→6、leuven 1→6；同时列出 OpenCV ORB 作为参照；
3. 在 bark 1→4、bark 1→6 上，取正确匹配，计算**尺度比**（`size2/size1` 的中位数）和**方向差**（`(θ2−θ1) mod 360` 的中位数），与真值比较。
   真值：$J=H_{[:2,:2]}/H_{22}$，尺度 $=\sqrt{\lvert\det J\rvert}$，旋转 $=\operatorname{atan2}(J_{10}, J_{00})$（度，mod 360）。OpenCV 的 `k.size`、`k.angle` 直接可用；你自己的实现用 `my_sift` 返回的 `sizes, angles`。

In [ ]:
def gaussian_pyramid(img, n_oct=5, s=3, sigma0=1.6):
    # TODO ①：返回 (pyr, sig)。pyr[o] 是形状 (s+3, h_o, w_o) 的 float32 数组；sig 是组内各层的 sigma 列表
    raise NotImplementedError

def dog_pyramid(pyr):
    # TODO ①
    raise NotImplementedError

def find_extrema(D, thr=0.5 * 0.04 / 3):
    # TODO ①：D 形状 (5, h, w)；返回 M x 3 的 int 数组 (s, y, x)，s ∈ {1,2,3}；离边界 5 像素内的点不要
    raise NotImplementedError

In [ ]:
# ===== ②③④ 及 my_sift：已给出，读懂后直接运行 =====
# ② refine：在 (x, y, sigma) 三个方向上用有限差分求一阶导 g 和 3x3 Hessian，
#    亚像素偏移 x̂ = -H^{-1} g；|x̂| > 0.5 说明真正的极值在邻点（此处简化为直接丢弃，不迭代）；
#    |D(x̂)| = |D + 0.5 g·x̂| 太小 → 低对比度，丢弃；2x2 空间 Hessian 的 tr²/det ≥ (r+1)²/r → 边缘，丢弃
# ③ orientation：36 bin 方向直方图，梯度幅值 × 高斯权投票，平滑后取峰值并抛物线插值（简化：不加 80% 辅方向）
# ④ descriptor：在旋转到主方向的坐标系里采 16x16 个点，每 4x4 个点为一个子块，统计 8 方向梯度直方图
#    → 128 维；归一化、截断 0.2（抑制非线性光照下个别大梯度）、再归一化
def refine(D, ext, contrast=0.04 / 3, r=10.0):
    s, y, x = ext.T
    d = lambda ds, dy, dx: D[s + ds, y + dy, x + dx]
    c = d(0, 0, 0)
    g = np.stack([(d(0, 0, 1) - d(0, 0, -1)) / 2, (d(0, 1, 0) - d(0, -1, 0)) / 2, (d(1, 0, 0) - d(-1, 0, 0)) / 2], 1)
    dxx = d(0, 0, 1) + d(0, 0, -1) - 2 * c; dyy = d(0, 1, 0) + d(0, -1, 0) - 2 * c; dss = d(1, 0, 0) + d(-1, 0, 0) - 2 * c
    dxy = (d(0, 1, 1) - d(0, 1, -1) - d(0, -1, 1) + d(0, -1, -1)) / 4
    dxs = (d(1, 0, 1) - d(1, 0, -1) - d(-1, 0, 1) + d(-1, 0, -1)) / 4
    dys = (d(1, 1, 0) - d(1, -1, 0) - d(-1, 1, 0) + d(-1, -1, 0)) / 4
    Hm = np.stack([np.stack([dxx, dxy, dxs], 1), np.stack([dxy, dyy, dys], 1), np.stack([dxs, dys, dss], 1)], 1).astype(np.float64)
    okH = np.abs(np.linalg.det(Hm)) > 1e-12
    off = np.zeros(g.shape, np.float64)
    off[okH] = -np.linalg.solve(Hm[okH], g[okH][..., None].astype(np.float64))[..., 0]
    val = c + 0.5 * (g * off).sum(1)                     # ② 极值处的响应
    tr = dxx + dyy; det = dxx * dyy - dxy ** 2           # ② 2x2 Hessian 去边缘
    edge_ok = (det > 0) & (tr ** 2 * r < (r + 1) ** 2 * np.where(det > 0, det, 1))
    keep = okH & (np.abs(off).max(1) <= 0.5) & (np.abs(val) > contrast) & edge_ok
    return ext[keep], off[keep], np.abs(val[keep])

def orientation(L, x, y, sigma, nb=36):
    r = int(round(3 * 1.5 * sigma)); h, w = L.shape
    x0, x1, y0, y1 = max(x - r, 1), min(x + r, w - 2), max(y - r, 1), min(y + r, h - 2)
    P = L[y0 - 1:y1 + 2, x0 - 1:x1 + 2]
    gx = P[1:-1, 2:] - P[1:-1, :-2]; gy = P[2:, 1:-1] - P[:-2, 1:-1]
    yy, xx = np.mgrid[y0:y1 + 1, x0:x1 + 1]
    wgt = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * (1.5 * sigma) ** 2))
    ang = np.arctan2(gy, gx) % (2 * np.pi)
    hist = np.bincount((ang / (2 * np.pi) * nb).astype(int).ravel() % nb, (np.hypot(gx, gy) * wgt).ravel(), nb)
    hist = np.convolve(np.r_[hist[-2:], hist, hist[:2]], [1, 4, 6, 4, 1], 'valid') / 16.0   # 平滑直方图
    i = int(np.argmax(hist)); l, c, rr = hist[i - 1], hist[i], hist[(i + 1) % nb]
    den = l - 2 * c + rr
    shift = 0.5 * (l - rr) / den if den != 0 else 0.0                                         # 抛物线插值峰值
    return ((i + 0.5 + shift) * 2 * np.pi / nb) % (2 * np.pi)

def descriptor(L, x, y, sigma, theta):
    bw = 3 * sigma                                         # 每个子块宽 3σ
    u = (np.arange(16) - 7.5) / 4.0 * bw
    U, V = np.meshgrid(u, u)
    c, s = np.cos(theta), np.sin(theta)
    mx = x + c * U - s * V; my = y + s * U + c * V          # 旋转到主方向的采样网格
    def samp(du, dv):
        return cv2.remap(L, (mx + c * du - s * dv).astype(np.float32), (my + s * du + c * dv).astype(np.float32),
                         cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    st = max(bw / 4.0, 1.0)
    gu = samp(st, 0) - samp(-st, 0); gv = samp(0, st) - samp(0, -st)   # 相对主方向的梯度
    mag = np.hypot(gu, gv) * np.exp(-(U ** 2 + V ** 2) / (2 * (2 * bw) ** 2))
    b = ((np.arctan2(gv, gu) % (2 * np.pi)) / (2 * np.pi) * 8).astype(int) % 8
    cell = np.arange(16) // 4
    idx = (cell[:, None] * 4 + cell[None, :]) * 8 + b
    v = np.bincount(idx.ravel(), mag.ravel(), 128)
    v /= np.linalg.norm(v) + 1e-12; v = np.minimum(v, 0.2); v /= np.linalg.norm(v) + 1e-12
    return v.astype(np.float32)

def my_sift(img, n_oct=5, s=3, max_kp=1000):
    pyr, sig = gaussian_pyramid(img, n_oct, s)
    dogs = dog_pyramid(pyr)
    cands = []
    for o, D in enumerate(dogs):
        ext = find_extrema(D)
        if len(ext) == 0:
            continue
        ext, off, resp = refine(D, ext)
        for (sl, y, x), (ox, oy, os_), rsp in zip(ext, off, resp):
            cands.append((o, sl, x + ox, y + oy, sig[0] * 2 ** ((sl + os_) / float(s)), rsp))
    cands.sort(key=lambda c: -c[5])
    cands = cands[:max_kp]
    pts, sizes, angs, desc = [], [], [], []
    for o, sl, x, y, sg, rsp in cands:
        L = pyr[o][sl]
        th = orientation(L, int(round(x)), int(round(y)), sg)
        desc.append(descriptor(L, x, y, sg, th))
        pts.append((x * 2 ** o, y * 2 ** o)); sizes.append(sg * 2 ** o); angs.append(th)
    return np.array(pts).reshape(-1, 2), np.array(sizes), np.array(angs), np.array(desc, np.float32).reshape(-1, 128)

In [ ]:
# ---- 中间结果检查（直接运行）：金字塔、DoG、每一步剩下多少点 ----
pyr, sig = gaussian_pyramid(g1); dogs = dog_pyramid(pyr)
print('组数', len(pyr), ' 每组层数', pyr[0].shape[0], ' 各组尺寸', [p.shape[1:] for p in pyr])
print('组内 sigma:', np.round(sig, 2))
n_ext = n_ref = 0
for D in dogs:
    e = find_extrema(D); n_ext += len(e)
    if len(e): n_ref += len(refine(D, e)[0])
print('① 26 邻域极值候选 %d 个 → ② 精确定位/去低对比/去边缘后 %d 个' % (n_ext, n_ref))
n_noedge = sum(len(refine(D, find_extrema(D), r=1e9)[0]) for D in dogs)
n_nocon = sum(len(refine(D, find_extrema(D), contrast=0.0)[0]) for D in dogs)
print('对照：② 中不去边缘(r 取极大) → %d 个；不去低对比度(contrast=0) → %d 个' % (n_noedge, n_nocon))
show([pyr[0][i] for i in range(0, 6, 2)] + [dogs[0][1] * 10 + 0.5], ['L sigma=%.2f' % sig[i] for i in range(0, 6, 2)] + ['DoG (x10)'])

In [ ]:
# TODO (C) 1：两种实现的关键点数、耗时，并画图

In [ ]:
# TODO (C) 2：7 组测试，自己的 SIFT / OpenCV SIFT / OpenCV ORB 的正确匹配数

In [ ]:
# TODO (C) 3：bark 1->4、1->6 上的尺度比与方向差 vs 真值（两种实现）

**问题 2**：(a) 自己实现的 SIFT 与 OpenCV SIFT 在关键点数、速度、各组测试的正确匹配数上差多少？读给出的 ②③④ 代码，指出它相对课件做了哪些简化；中间结果检查中，去边缘、去低对比度分别去掉了多少点？为什么要去掉它们？
(b) 两种实现测得的尺度比、方向差与真值差多少？这说明 SIFT 的哪两步确实把"尺度"和"方向"算对了？
(c) bark 1→6（约 4 倍缩放）上 SIFT 与 ORB 谁更好？用 ORB 默认金字塔（`nlevels=8, scaleFactor=1.2`，覆盖 $1.2^7\approx3.6$ 倍）解释。
(d) graf 1→5 上三者都几乎没有正确匹配。SIFT 号称尺度、旋转不变，为什么这里也不行？它对哪类变化没有不变性？
(e) leuven 1→6 上匹配表现如何？为什么图像整体变暗后匹配几乎不受影响（SIFT 描述子的哪些设计起了作用）？

★ 选做：给 `refine` 加上迭代（偏移 > 0.5 时移到邻点重新展开，最多 5 次），或在 `orientation` 中加入 80% 辅方向，看看正确匹配数是否提高。

> 你的回答：

## 任务 5　总结

1. 结合本课实验结果和课件，填写下表（✓ / △ / ✗；有实验数据的格子请在括号中写出支撑它的一个数字）：

| 方法 | 旋转 | 尺度 | 光照变暗 | 大视角 |
|---|---|---|---|---|
| FAST | | | | |
| Harris | | | | |
| BRIEF | | | | |
| ORB | | | | |
| SIFT | | | | |

2. Harris 和 SIFT：你自己的实现与 OpenCV 的主要差别分别是什么？各用一个数字说明。

> 你的回答：